In [6]:
!pip install google-genai opencv-python pillow numpy scikit-learn easyocr pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 20.6 MB/s eta 0:00:00


In [7]:
from google.colab import userdata
api_key= userdata.get('GEMINI_API_KEY')

In [12]:
import json
import os
import cv2
import easyocr
import numpy as np
from google import genai
from google.genai import types
from PIL import Image
from pydantic import BaseModel, Field
from sklearn.cluster import KMeans
from urllib.parse import urlparse

# ==========================================
# 1. Pydantic Schemas for Gemini Structured Output
# ==========================================
class BrandSafety(BaseModel):
    is_safe: bool = Field(
        description="False if explicit content, violence, hate symbols, or high visual risk is present."
    )
    flags: list[str] = Field(
        description="Specific flags raised (e.g., 'adult', 'medical', 'weapons', 'brand_risk'). Empty if clean."
    )


class TargetAudience(BaseModel):
    perceived_age_group: str = Field(
        description="Target demographic (e.g., 'Gen Z', 'Young Professionals', 'Parents', 'Seniors')"
    )
    lifestyle_vibe: str = Field(
        description="Visual aesthetic style (e.g., 'Minimalist', 'High-energy Luxury', 'Casual Outdoor')"
    )


class HighLevelMarketingFeatures(BaseModel):
    product_category: str = Field(
        description="Primary product/service category (e.g., 'Footwear', 'SaaS Platform', 'Skincare')"
    )
    secondary_tags: list[str] = Field(
        description="Sub-categories, context tags, or product features observed."
    )
    perceived_emotion: str = Field(
        description="Dominant emotional appeal (e.g., 'Joy', 'Urgency', 'Calm/Trust', 'Excitement')"
    )
    brand_logos: list[str] = Field(
        description="Visible logos, brand names, or identifiable trade marks detected."
    )
    brand_safety: BrandSafety
    target_audience: TargetAudience
    value_proposition_summary: str = Field(
        description="One-sentence summary of what the ad or image visually communicates."
    )


# ==========================================
# 2. Unified Extractor (OpenCV + Gemini Free Tier)
# ==========================================
class GeminiMarketingExtractorSingleImage:

    def __init__(self, url:str, api_key: str | None = None):
        """Initializes OpenCV/OCR and the Google GenAI Client."""
        # Initialize EasyOCR once
        self.url = url
        self.ocr_reader = easyocr.Reader(["en"], gpu=False)

        # Initialize Google GenAI Client
        api_key = api_key or os.environ.get("GEMINI_API_KEY")
        if not api_key:
            raise ValueError(
                "GEMINI_API_KEY environment variable or parameter is required."
            )

        self.ai_client = genai.Client(api_key=api_key)

    def extract_brand_and_category(self):
      # 1. Parse the URL components
        parsed_url = urlparse(self.url)

        # 2. Extract 'ing' from the domain (netloc)
        # e.g., 'www.ing.be' -> splits by '.' -> gets 'ing'
        domain_parts = parsed_url.netloc.split(".")
        brand = domain_parts[1] if len(domain_parts) > 1 else domain_parts[0]

        # 3. Extract the credit card category from the path segments
        # Split path by '/' and filter out empty strings
        path_segments = [seg for seg in parsed_url.path.split("/") if seg]
        # path_segments will be: ['fr', 'particuliers', 'cartes-de-credit', 'carte-de-credit-visa']

        # You can grab specific indices or search for keywords
        category = path_segments[2] if len(path_segments) > 2 else None
        full_card_path = path_segments[3] if len(path_segments) > 3 else None

        print(f"Brand: {brand}")
        print(f"Category: {category}")
        print(f"Full Card Path: {full_card_path}")
        return brand, category

    def _extract_low_level_features(
        self, image_path: str, num_colors: int = 3
    ) -> dict:
        """Extracts low-level computer vision metrics via OpenCV & EasyOCR."""
        img_bgr = cv2.imread(image_path)
        if img_bgr is None:
            raise ValueError(f"Could not load image at path: {image_path}")

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        height, width, _ = img_rgb.shape

        # 1. Structural Dimensions
        aspect_ratio = round(width / height, 2)

        # 2. Color & Tone Metrics
        brightness = round(float(np.mean(img_hsv[:, :, 2])), 2)
        saturation = round(float(np.mean(img_hsv[:, :, 1])), 2)

        # Dominant Colors via K-Means Clustering
        pixels = img_rgb.reshape(-1, 3)
        kmeans = KMeans(n_clusters=num_colors, n_init=5, random_state=42).fit(
            pixels
        )
        hex_colors = [
            f"#{r:02x}{g:02x}{b:02x}"
            for r, g, b in kmeans.cluster_centers_.astype(int)
        ]

        # 3. Contrast & Visual Complexity
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        contrast_rms = round(float(gray.std()), 2)

        edges = cv2.Canny(gray, 100, 200)
        edge_density = float(np.sum(edges > 0)) / (height * width)

        # 4. Text Overlay & Coverage (OCR)
        ocr_results = self.ocr_reader.readtext(image_path)
        total_text_area = 0
        detected_texts = []

        for bbox, text, prob in ocr_results:
            if prob > 0.3:
                detected_texts.append(text)
                pt1, pt2, pt3, pt4 = bbox
                box_w = max(pt2[0], pt3[0]) - min(pt1[0], pt4[0])
                box_h = max(pt3[1], pt4[1]) - min(pt1[1], pt2[1])
                total_text_area += box_w * box_h

        text_coverage_pct = round((total_text_area / (width * height)) * 100, 2)

        return {
            "dimensions": {"width": width, "height": height},
            "aspect_ratio": aspect_ratio,
            "brightness": brightness,
            "saturation": saturation,
            "dominant_colors": hex_colors,
            "contrast_rms": contrast_rms,
            "visual_clutter_score": round(edge_density, 4),
            "text_coverage_pct": text_coverage_pct,
            "ocr_copy_count": len(detected_texts),
            "ocr_detected_copy": detected_texts,
        }

    def _extract_gemini_features(
        self, image_path: str
    ) -> HighLevelMarketingFeatures:
        """Extracts high-level marketing attributes via Gemini (Free Tier Model) using Pydantic schema enforcement."""
        pil_image = Image.open(image_path)
        brand, category = self.extract_brand_and_category()
        prompt = (
            "Analyze this marketing image. Categorize the product, "
            "assess brand safety, detect logos, and analyze emotional tone, "
            f"value proposition, and target demographic appeal in the context of {brand} and product category {category}"
        )

        # Call Gemini using Structured Output (gemini-2.5-flash or gemini-3-flash)
        response = self.ai_client.models.generate_content(
            model="gemini-3.6-flash",  # 100% Free Tier Model
            contents=[pil_image, prompt],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=HighLevelMarketingFeatures,
            ),
        )

        # Parse JSON output into Pydantic model
        parsed_json = json.loads(response.text)
        return HighLevelMarketingFeatures(**parsed_json)

    def analyze_image(self, image_path: str) -> dict:
        """Pipeline entry point combining local CV metrics and Gemini Free Vision analysis."""
        cv_metrics = self._extract_low_level_features(image_path)
        gemini_analysis = self._extract_gemini_features(image_path)

        return {
            "low_level_metrics": cv_metrics,
            "semantic_analysis": gemini_analysis.model_dump(),
        }


# ==========================================
# 3. Execution Script
# ==========================================
if __name__ == "__main__":
    sample_image = "visa_classic_ing.jpg"
    url = "https://www.ing.be/fr/particuliers/cartes-de-credit/carte-de-credit-visa"
    extractor = GeminiMarketingExtractorSingleImage(url=url,api_key=api_key)

    try:
        report = extractor.analyze_image(sample_image)
        print("=== UNIFIED MARKETING REPORT (GEMINI FREE TIER) ===")
        print(json.dumps(report, indent=2))

    except Exception as e:
        print(f"Pipeline execution failed: {e}")

Brand: ing
Category: cartes-de-credit
Full Card Path: carte-de-credit-visa
=== UNIFIED MARKETING REPORT (GEMINI FREE TIER) ===
{
  "low_level_metrics": {
    "dimensions": {
      "width": 1440,
      "height": 480
    },
    "aspect_ratio": 3.0,
    "brightness": 182.1,
    "saturation": 160.04,
    "dominant_colors": [
      "#289cc0",
      "#beaa8a",
      "#493526"
    ],
    "contrast_rms": 37.87,
    "visual_clutter_score": 0.0197,
    "text_coverage_pct": 0.0,
    "ocr_copy_count": 0,
    "ocr_detected_copy": []
  },
  "semantic_analysis": {
    "product_category": "Financial Services",
    "secondary_tags": [
      "Hiking",
      "Outdoor Adventure",
      "Empowerment",
      "Fitness",
      "Credit Cards"
    ],
    "perceived_emotion": "Empowerment",
    "brand_logos": [],
    "brand_safety": {
      "is_safe": true,
      "flags": []
    },
    "target_audience": {
      "perceived_age_group": "Young Professionals",
      "lifestyle_vibe": "Active Outdoor"
    },
    "va